In [1]:
%pwd

'd:\\Siam\\KIdney disease\\research'

In [2]:
import os

In [3]:
os.chdir("../")

In [4]:
%pwd

'd:\\Siam\\KIdney disease'

In [5]:
from dataclasses import dataclass
from pathlib import Path

@dataclass(frozen=True)
class EvaluationConfig:
    root_dir: Path
    dataset_dir: Path
    model_path: Path
    report_file: Path
    confusion_matrix_file: Path

In [6]:
from kidney_disease_classification.constants import *
from kidney_disease_classification.utils.common import read_yaml, create_directories

In [7]:
class ConfigurationManager:
    def __init__(
        self,
        config_filepath = CONFIG_FILE_PATH,
        params_filepath = PARAMS_FILE_PATH):

        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)

        create_directories([self.config["artifacts_root"]])

    def get_evaluation_config(self) -> EvaluationConfig:
        config = self.config["evaluation"]

        create_directories([config["root_dir"]])

        evaluation_config = EvaluationConfig(
            root_dir = config["root_dir"],
            dataset_dir = config["dataset_dir"],
            model_path = config["model_path"],
            report_file = config["report_file"],
            confusion_matrix_file = config["confusion_matrix_file"]
        )

        return evaluation_config

In [9]:
import sys
import torch
import torch.nn as nn
from pathlib import Path
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix
)

from kidney_disease_classification.exception import CustomException
from kidney_disease_classification.logger import logging
from kidney_disease_classification.utils.common import save_json

In [10]:
class Evaluation:
    def __init__(self, config: EvaluationConfig, params):
        self.config = config
        self.params = params
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    # ---------------------------
    # LOAD MODEL
    # ---------------------------
    def load_model(self):
        try:
            model = models.efficientnet_b0(pretrained=False)

            in_features = model.classifier[1].in_features
            model.classifier[1] = nn.Linear(in_features, self.params["NUM_CLASSES"])

            model.load_state_dict(
                torch.load(self.config.model_path, map_location=self.device)
            )

            model.to(self.device)
            model.eval()

            logging.info("Model loaded successfully for evaluation.")
            return model

        except Exception as e:
            raise CustomException(e, sys)

    # ---------------------------
    # LOAD TEST DATA
    # ---------------------------
    def get_test_loader(self):
        try:
            img_size = tuple(self.params["IMAGE_SIZE"])

            test_transform = transforms.Compose([
                transforms.Resize(img_size),
                transforms.Grayscale(num_output_channels=3),
                transforms.ToTensor(),
                transforms.Normalize([0.5, 0.5, 0.5], [0.5, 0.5, 0.5])
            ])

            test_dir = Path(self.config.dataset_dir) / "test"

            test_dataset = datasets.ImageFolder(test_dir, transform=test_transform)

            test_loader = DataLoader(
                test_dataset,
                batch_size=self.params["BATCH_SIZE"],
                shuffle=False
            )

            logging.info("Test loader created successfully.")
            return test_loader

        except Exception as e:
            raise CustomException(e, sys)

    # ---------------------------
    # MAIN EVALUATION FUNCTION
    # ---------------------------
    def evaluate_model(self):
        try:
            logging.info("Evaluation stage started...")

            model = self.load_model()
            test_loader = self.get_test_loader()

            y_true = []
            y_pred = []
            y_probs = []

            with torch.no_grad():
                for images, labels in test_loader:
                    images = images.to(self.device)
                    labels = labels.to(self.device)

                    outputs = model(images).squeeze()
                    probs = torch.sigmoid(outputs)
                    preds = (probs > 0.5).int()

                    y_true.extend(labels.cpu().numpy())
                    y_pred.extend(preds.cpu().numpy())
                    y_probs.extend(probs.cpu().numpy())

            # Metrics
            acc = accuracy_score(y_true, y_pred)
            precision = precision_score(y_true, y_pred)
            recall = recall_score(y_true, y_pred)
            f1 = f1_score(y_true, y_pred)
            auc = roc_auc_score(y_true, y_probs)

            cm = confusion_matrix(y_true, y_pred)
            tn, fp, fn, tp = cm.ravel()

            specificity = tn / (tn + fp)

            report = {
                "accuracy": float(acc),
                "precision": float(precision),
                "recall_sensitivity": float(recall),
                "specificity": float(specificity),
                "f1_score": float(f1),
                "roc_auc": float(auc),
                "tp": int(tp),
                "tn": int(tn),
                "fp": int(fp),
                "fn": int(fn)
            }

            # Save evaluation report
            save_json(self.config.report_file, report)

            # Save confusion matrix separately
            save_json(self.config.confusion_matrix_file, {
                "confusion_matrix": cm.tolist()
            })

            logging.info("Evaluation completed successfully.")
            logging.info(f"Evaluation Report: {report}")

            return report

        except Exception as e:
            raise CustomException(e, sys)

In [12]:
try:
    config = ConfigurationManager()
    evaluation_config = config.get_evaluation_config()
    evaluation = Evaluation(config=evaluation_config, params=config.params)
    evaluation_report = evaluation.evaluate_model()
except Exception as e:
    raise CustomException(e, sys)

[2026-05-14 01:50:36,942: INFO: common: yaml file: config\config.yaml loaded successfully]
[2026-05-14 01:50:36,946: INFO: common: yaml file: params.yaml loaded successfully]
[2026-05-14 01:50:36,948: INFO: common: created directory at: artifacts]
[2026-05-14 01:50:36,950: INFO: common: created directory at: artifacts/evaluation]
[2026-05-14 01:50:36,951: INFO: 2958062136: Evaluation stage started...]
[2026-05-14 01:50:37,167: INFO: 2958062136: Model loaded successfully for evaluation.]
[2026-05-14 01:50:37,169: INFO: 2958062136: Test loader created successfully.]
[2026-05-14 01:50:40,395: INFO: common: json file saved at: artifacts/evaluation/evaluation_report.json]
[2026-05-14 01:50:40,397: INFO: common: json file saved at: artifacts/evaluation/confusion_matrix.json]
[2026-05-14 01:50:40,398: INFO: 2958062136: Evaluation completed successfully.]
[2026-05-14 01:50:40,399: INFO: 2958062136: Evaluation Report: {'accuracy': 0.7887323943661971, 'precision': 0.9473684210526315, 'recall_sen